# Week 6: ViT Wall-Clock Optimization for CIFAR-10

## Goal

Reduce wall-clock time to reach 94% test accuracy by applying:
1. **Mixed precision (AMP)** - FP16 on CUDA for ~1.5-2x throughput
2. **LR scheduler** - Cosine annealing with warmup for faster convergence
3. **Deterministic toggle** - `cudnn.benchmark=True` for speed (vs reproducibility)
4. **RandAugment** - Stronger augmentation (optional)
5. **DataLoader tuning** - `persistent_workers`, `prefetch_factor`

Primary metric: **time-to-target** (seconds to reach 94%).

## Imports and Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"

import torch
from src.model import VisionTransformer, count_parameters
from src.utils import (
    get_device,
    get_cifar10_loaders,
    get_model,
    get_pretrained_vit,
    set_seed,
    train,
    reset_peak_gpu_memory,
)

print(f"PyTorch: {torch.__version__}")
device = get_device()
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

**Speed mode**: `DETERMINISTIC=False` enables `cudnn.benchmark` (faster).  
**AMP**: Uses mixed precision on CUDA (MPS/CPU fall back to FP32).  
**Scheduler**: Cosine annealing with 3-epoch warmup.

In [ ]:
SEED = 42
TARGET_ACCURACY = 94.0
MAX_EPOCHS = 100

# Speed vs reproducibility: False = faster (cudnn.benchmark)
DETERMINISTIC = False
set_seed(SEED, deterministic=DETERMINISTIC)

# Optimizations
USE_PRETRAINED = True
USE_AMP = True
USE_SCHEDULER = True
WARMUP_EPOCHS = 3
USE_RANDAUGMENT = False  # Set True to try stronger augmentation

# Data loading
BATCH_SIZE = 64 if USE_PRETRAINED else 128
NUM_WORKERS = 4
PERSISTENT_WORKERS = True

# Training
LR = 3e-4
IMG_SIZE = 224 if USE_PRETRAINED else 32

print(f"DETERMINISTIC={DETERMINISTIC} | AMP={USE_AMP} | Scheduler={USE_SCHEDULER}")
print(f"RandAugment={USE_RANDAUGMENT} | persistent_workers={PERSISTENT_WORKERS}")

## Data Loading

CIFAR-10 with optional RandAugment. DataLoader uses `persistent_workers` and `prefetch_factor` when workers > 0.

In [ ]:
train_loader, test_loader = get_cifar10_loaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment_train=True,
    img_size=IMG_SIZE,
    use_randaugment=USE_RANDAUGMENT,
    randaugment_num_ops=2,
    randaugment_magnitude=9,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=2,
)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
sample_batch, sample_labels = next(iter(train_loader))
print(f"Batch shape: {sample_batch.shape}")

## Model Creation

In [ ]:
if USE_PRETRAINED:
    model = get_pretrained_vit(
        model_name="vit_tiny_patch16_224",
        num_classes=10,
        pretrained=True,
    )
else:
    model = get_model(
        patch_size=4,
        embed_dim=96,
        depth=4,
        num_heads=4,
        dropout=0.0,
    )

print(f"Parameters: {count_parameters(model):,}")
reset_peak_gpu_memory()
model = model.to(device)
_ = model(sample_batch.to(device))
print(f"Output shape: {_.shape}")

## Training Loop (with AMP + Scheduler)

In [ ]:
reset_peak_gpu_memory()

results = train(
    model,
    train_loader,
    test_loader,
    num_epochs=MAX_EPOCHS,
    lr=LR,
    device=device,
    log_every=1,
    target_accuracy=TARGET_ACCURACY,
    use_amp=USE_AMP,
    use_scheduler=USE_SCHEDULER,
    warmup_epochs=WARMUP_EPOCHS,
)

ttt = results.get("time_to_target")
print("\n--- Summary ---")
print(f"Best test accuracy: {results['best_acc']:.2f}%")
print(f"Time-to-target ({TARGET_ACCURACY}%): {ttt:.2f}s" if ttt else f"Time-to-target: Not reached")
print(f"Total wall-clock: {results['total_time']:.2f}s")
print(f"Peak GPU memory: {results.get('peak_gpu_mb', 'N/A')} MB")

## Accuracy and Throughput Curves

In [ ]:
import matplotlib.pyplot as plt

history = results["history"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_acc"], label="Train")
axes[0].plot(history["test_acc"], label="Test")
axes[0].axhline(y=TARGET_ACCURACY, color="gray", linestyle="--", label=f"Target {TARGET_ACCURACY}%")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Accuracy over time")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["throughput"], color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Images/sec")
axes[1].set_title("Training throughput")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Comparison: Baseline vs Optimized

To compare, re-run this notebook with different configs and record:
- **Baseline**: `DETERMINISTIC=True`, `USE_AMP=False`, `USE_SCHEDULER=False`
- **Optimized**: `DETERMINISTIC=False`, `USE_AMP=True`, `USE_SCHEDULER=True`

Expected: optimized run shows faster time-to-target and higher throughput on CUDA.